<a href="https://colab.research.google.com/github/islam37/S-SLTE-Thesis/blob/main/notebooks/05_symmetric_noise_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import numpy as np
import pandas as pd

SEED = 42
NOISE_RATE = 0.218
NUM_CLASSES = 4

def inject_symmetric_noise(
    clean_labels,
    noise_rate,
    seed=42,
    num_classes=4
):
    labels = np.asarray(clean_labels, dtype=int)

    assert labels.ndim == 1
    assert 0 <= noise_rate <= 1
    assert num_classes >= 2
    assert np.all(
        (labels >= 0) & (labels < num_classes)
    )

    rng = np.random.default_rng(seed)
    noisy = labels.copy()

    # Fixed-count corruption, approximately noise_rate
    n_flip = round(len(labels) * noise_rate)

    flip_idx = rng.choice(
        len(labels),
        size=n_flip,
        replace=False
    )

    # Choose uniformly among incorrect classes
    offsets = rng.integers(
        1,
        num_classes,
        size=n_flip
    )

    noisy[flip_idx] = (
        labels[flip_idx] + offsets
    ) % num_classes

    return noisy, flip_idx


# Synthetic data only
clean = np.tile(np.arange(4), 125)

noisy, changed_indices = inject_symmetric_noise(
    clean,
    noise_rate=NOISE_RATE,
    seed=SEED,
    num_classes=NUM_CLASSES
)

# Unit tests
assert len(clean) == 500
assert len(changed_indices) == 109

assert np.sum(clean != noisy) == 109
assert np.all(
    clean[changed_indices] != noisy[changed_indices]
)

# Reproducibility test
noisy_again, indices_again = inject_symmetric_noise(
    clean, NOISE_RATE, SEED, NUM_CLASSES
)

assert np.array_equal(noisy, noisy_again)
assert np.array_equal(
    changed_indices,
    indices_again
)

# Invalid input test
try:
    inject_symmetric_noise(clean, 1.5)
    raise AssertionError("Invalid rate accepted")
except AssertionError as exc:
    assert str(exc) != "Invalid rate accepted"

print("SYMMETRIC NOISE GENERATOR TEST PASSED")
print("-------------------------------------")
print("Synthetic records:", len(clean))
print("Noise target:", NOISE_RATE)
print("Changed labels:", np.sum(clean != noisy))
print(
    "Actual change rate:",
    f"{np.mean(clean != noisy)*100:.2f}%"
)
print("Reproducibility: PASSED")
print("No self-label transitions: PASSED")
print("Real dataset used: NO")

matrix = pd.crosstab(
    pd.Series(clean, name="Original"),
    pd.Series(noisy, name="Noisy")
)

display(matrix)


SYMMETRIC NOISE GENERATOR TEST PASSED
-------------------------------------
Synthetic records: 500
Noise target: 0.218
Changed labels: 109
Actual change rate: 21.80%
Reproducibility: PASSED
No self-label transitions: PASSED
Real dataset used: NO


Noisy,0,1,2,3
Original,,,,
0,93,11,15,6
1,6,100,7,12
2,4,7,104,10
3,13,8,10,94


In [2]:

import numpy as np
import pandas as pd

# Uses inject_symmetric_noise() from Step 1

NUM_RUNS = 200
SEED_START = 1000
NUM_CLASSES = 4
NOISE_RATE = 0.218

# Balanced synthetic input
clean = np.tile(np.arange(NUM_CLASSES), 125)

all_offsets = []
all_changes = []

for seed in range(SEED_START, SEED_START + NUM_RUNS):

    noisy, flipped_indices = inject_symmetric_noise(
        clean,
        noise_rate=NOISE_RATE,
        seed=seed,
        num_classes=NUM_CLASSES
    )

    changed = (clean != noisy)

    # Every run must corrupt exactly 109 labels
    assert changed.sum() == 109
    assert len(flipped_indices) == 109

    # All resulting labels must be valid
    assert np.all((noisy >= 0) & (noisy < 4))

    # Reproducibility for identical seeds
    repeated, repeated_indices = inject_symmetric_noise(
        clean,
        NOISE_RATE,
        seed,
        NUM_CLASSES
    )

    assert np.array_equal(noisy, repeated)
    assert np.array_equal(
        flipped_indices, repeated_indices
    )

    # Relative offset identifies one of 3 wrong classes
    offsets = (
        noisy[changed] - clean[changed]
    ) % NUM_CLASSES

    assert np.all(np.isin(offsets, [1, 2, 3]))

    all_offsets.extend(offsets.tolist())
    all_changes.append(int(changed.sum()))

# Analyze uniformity across all 200 runs
counts = np.bincount(
    all_offsets, minlength=NUM_CLASSES
)[1:]

total_flips = sum(all_changes)
probabilities = counts / total_flips

summary = pd.DataFrame({
    "incorrect_class_offset": [1, 2, 3],
    "count": counts,
    "observed_probability": probabilities,
    "expected_probability": [1/3] * 3
})

assert total_flips == 200 * 109

# Diagnostic tolerance for this synthetic unit test.
# Not a formal statistical hypothesis test.
MAX_DEVIATION = 0.035

max_deviation = np.max(
    np.abs(probabilities - 1/3)
)

assert max_deviation < MAX_DEVIATION, (
    "Uniformity diagnostic exceeded tolerance"
)

print("SYMMETRIC NOISE FAIRNESS TEST PASSED")
print("-----------------------------------")
print("Independent runs:", NUM_RUNS)
print("Synthetic samples per run:", len(clean))
print("Total changed labels:", total_flips)
print(
    "Average corruption rate:",
    f"{np.mean(all_changes)/len(clean)*100:.2f}%"
)
print(
    "Maximum probability deviation:",
    f"{max_deviation:.4f}"
)

print("\nIncorrect-class selection distribution:")
display(summary)

print("Deterministic reproducibility: PASSED")
print("Uniformity diagnostic: PASSED")
print("Real reference labels used: NO")


SYMMETRIC NOISE FAIRNESS TEST PASSED
-----------------------------------
Independent runs: 200
Synthetic samples per run: 500
Total changed labels: 21800
Average corruption rate: 21.80%
Maximum probability deviation: 0.0069

Incorrect-class selection distribution:


,incorrect_class_offset,count,observed_probability,expected_probability
0,1,7170,0.328899,0.333333
1,2,7418,0.340275,0.333333
2,3,7212,0.330826,0.333333


Deterministic reproducibility: PASSED
Uniformity diagnostic: PASSED
Real reference labels used: NO
